## Convert stock-out risk + AI demand forecast into clear reorder recommendations
## (what to order, how much, and why)

### Store should reorder when Predicted demand during lead time + safety buffer
###     >
### Current stock

- ### We will compute:
- ### 
- ### Reorder Point (ROP)
- ### 
- ### Recommended Order Quantity
- ### 
- ### Action Flag (REORDER / OK)


🧱 OUTPUT TABLE (FINAL GOLD)

Table: inventory_ai.gold_replenishment_recommendations

- Schema
- date
- store_id
- product_family
- current_stock
- predicted_daily_demand
- lead_time_days
- reorder_point
- recommended_order_qty
- stockout_risk_level
- replenishment_action

In [0]:
spark.sql("USE inventory_ai")

risk_df = spark.table("gold_stockout_risk")


  Define Safety Stock Logic

  We’ll use risk-aware safety stock (this is AI insight).

- Risk Level	Safety Stock
- HIGH	30% extra
- MEDIUM	15% extra
- LOW	5% extra

In [0]:
from pyspark.sql.functions import col, when

risk_df = risk_df.withColumn(
    "safety_multiplier",
    when(col("stockout_risk_level") == "HIGH", 1.30)
    .when(col("stockout_risk_level") == "MEDIUM", 1.15)
    .otherwise(1.05)
)


Calculate Reorder Point (ROP)

In [0]:
risk_df = risk_df.withColumn(
    "reorder_point",
    col("predicted_daily_demand") *
    col("lead_time_days") *
    col("safety_multiplier")
)

Calculate Recommended Order Quantity

In [0]:
risk_df = risk_df.withColumn(
    "recommended_order_qty",
    when(
        col("current_stock") < col("reorder_point"),
        col("reorder_point") - col("current_stock")
    ).otherwise(0)
)


Replenishment Decision Flag

In [0]:
risk_df = risk_df.withColumn(
    "replenishment_action",
    when(col("recommended_order_qty") > 0, "REORDER")
    .otherwise("OK")
)


Save Final Gold Table

In [0]:
risk_df.select(
    "date",
    "store_id",
    "product_family",
    "current_stock",
    "predicted_daily_demand",
    "lead_time_days",
    "reorder_point",
    "recommended_order_qty",
    "stockout_risk_level",
    "replenishment_action"
).write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("inventory_ai.gold_replenishment_recommendations")


Validate Output

In [0]:
spark.sql("""
SELECT store_id, product_family, recommended_order_qty, replenishment_action
FROM gold_replenishment_recommendations
WHERE replenishment_action = 'REORDER'
ORDER BY recommended_order_qty DESC
LIMIT 10
""").show()
